In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/roblexnana/alice-wonderland-dataset/alice_in_wonderland.txt


In [2]:
from collections import Counter
import re

In [3]:
with open("/kaggle/input/datasets/roblexnana/alice-wonderland-dataset/alice_in_wonderland.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

In [4]:
text[:900]

"alice's adventures in wonderland\n\n                alice's adventures in wonderland\n\n                          lewis carroll\n\n               the millennium fulcrum edition 3.0\n\n\n\n\n                            chapter i\n\n                      down the rabbit-hole\n\n\n  alice was beginning to get very tired of sitting by her sister\non the bank, and of having nothing to do:  once or twice she had\npeeped into the book her sister was reading, but it had no\npictures or conversations in it, `and what is the use of a book,'\nthought alice `without pictures or conversation?'\n\n  so she was considering in her own mind (as well as she could,\nfor the hot day made her feel very sleepy and stupid), whether\nthe pleasure of making a daisy-chain would be worth the trouble\nof getting up and picking the daisies, when suddenly a white\nrabbit with pink eyes ran close by her.\n\n  there was nothing so very remarkable"

In [5]:
text = re.sub(r"[^a-z\s]", "", text)
words = text.split()

corpus = Counter(words)

print("Number of unique words:", len(corpus))
print("Most common words:", corpus.most_common(10))

Number of unique words: 2749
Most common words: [('the', 1632), ('and', 845), ('to', 721), ('a', 627), ('she', 537), ('it', 526), ('of', 508), ('said', 462), ('i', 401), ('alice', 386)]


In [6]:
def build_initial_vocab(corpus):

    vocab = {}

    for word, freq in corpus.items():
        
        chars = list(word)

        chars.append("</w>")

        token = " ".join(chars)

        vocab[token] = freq

    return vocab
        

In [7]:
vocab = build_initial_vocab(corpus)

print("Number of vocabulary entries:", len(vocab))

for i, (k, v) in enumerate(vocab.items()):
    print(f"{k} -> {v}")
    if i == 4:
        break

Number of vocabulary entries: 2749
a l i c e s </w> -> 13
a d v e n t u r e s </w> -> 7
i n </w> -> 367
w o n d e r l a n d </w> -> 4
l e w i s </w> -> 1


In [8]:
n = 20

for i, (key, value) in enumerate(vocab.items()):
    if i == n:
        break
    print(key)

a l i c e s </w>
a d v e n t u r e s </w>
i n </w>
w o n d e r l a n d </w>
l e w i s </w>
c a r r o l l </w>
t h e </w>
m i l l e n n i u m </w>
f u l c r u m </w>
e d i t i o n </w>
c h a p t e r </w>
i </w>
d o w n </w>
r a b b i t h o l e </w>
a l i c e </w>
w a s </w>
b e g i n n i n g </w>
t o </w>
g e t </w>
v e r y </w>


In [9]:
def get_pair_frequencies(vocab):

    pair_freqs = {}

    for word, freq in vocab.items():

        symbols = word.split()

        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])

            pair_freqs[pair] = pair_freqs.get(pair, 0) + freq

    return pair_freqs

In [10]:
pair_freqs = get_pair_frequencies(vocab)

sorted_pairs = sorted(pair_freqs.items(), key=lambda x: x[1], reverse=True)

print("Top 20 most frequent symbol pairs:")
for pair, freq in sorted_pairs[:20]:
    print(pair, "->", freq)

Top 20 most frequent symbol pairs:
('e', '</w>') -> 5734
('h', 'e') -> 3780
('t', 'h') -> 3484
('t', '</w>') -> 3269
('d', '</w>') -> 3210
('s', '</w>') -> 2214
('i', 'n') -> 2027
('e', 'r') -> 1823
('n', '</w>') -> 1795
('a', 'n') -> 1608
('o', 'u') -> 1556
('r', '</w>') -> 1506
('y', '</w>') -> 1417
('i', 't') -> 1325
('o', '</w>') -> 1314
('n', 'd') -> 1272
('a', 't') -> 1167
('r', 'e') -> 1151
('h', 'a') -> 1148
('n', 'g') -> 1140


In [11]:
def merge_vocab(pair, vocab):

    new_vocab = {}

    for word, freq in vocab.items():
        symbols = word.split()
        merged_symbols = []

        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1 and
                symbols[i] == pair[0] and
                symbols[i + 1] == pair[1]
            ):
                merged_symbols.append(pair[0] + pair[1])
                i += 2
            else:
                merged_symbols.append(symbols[i])
                i += 1

        new_word = " ".join(merged_symbols)
        new_vocab[new_word] = freq

    return new_vocab

In [12]:
pair_freqs = get_pair_frequencies(vocab)
best_pair = max(pair_freqs, key=pair_freqs.get)

print("Most frequent pair:", best_pair)

vocab = merge_vocab(best_pair, vocab)

for i, (k, v) in enumerate(vocab.items()):
    print(f"{k} -> {v}")
    if i == 20:
        break

Most frequent pair: ('e', '</w>')
a l i c e s </w> -> 13
a d v e n t u r e s </w> -> 7
i n </w> -> 367
w o n d e r l a n d </w> -> 4
l e w i s </w> -> 1
c a r r o l l </w> -> 1
t h e</w> -> 1632
m i l l e n n i u m </w> -> 1
f u l c r u m </w> -> 1
e d i t i o n </w> -> 1
c h a p t e r </w> -> 12
i </w> -> 401
d o w n </w> -> 101
r a b b i t h o l e</w> -> 3
a l i c e</w> -> 386
w a s </w> -> 357
b e g i n n i n g </w> -> 13
t o </w> -> 721
g e t </w> -> 46
v e r y </w> -> 144
t i r e d </w> -> 7


In [13]:
def train_bpe(corpus, num_merges):

    vocab = build_initial_vocab(corpus)

    merges = []

    for i in range(num_merges):

        pair_freqs = get_pair_frequencies(vocab)

        if not pair_freqs:
            break

        best_pair = max(pair_freqs, key=pair_freqs.get)

        vocab = merge_vocab(best_pair, vocab)

        merges.append(best_pair)

    return vocab, merges

In [14]:
toy = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
vocab, merges = train_bpe(toy, num_merges=10)

print("Merges:", merges[:5])
print("Sample vocab:")
for i, (k, v) in enumerate(vocab.items()):
    print(k, "->", v)
    if i == 4:
        break

Merges: [('e', 's'), ('es', 't'), ('est', '</w>'), ('l', 'o'), ('lo', 'w')]
Sample vocab:
low</w> -> 5
low e r </w> -> 2
newest</w> -> 6
wi d est</w> -> 3


In [15]:
def tokenize_word_bpe(word, merges):

    tokens = list(word) + ["</w>"]

    for pair in merges:
        merged_tokens = []

        i = 0
        while i < len(tokens):
            if (
                i < len(tokens) - 1 and
                tokens[i] == pair[0] and
                tokens[i + 1] == pair[1]
            ):
                merged_tokens.append(pair[0] + pair[1])
                i += 2
            else:
                merged_tokens.append(tokens[i])
                i += 1

        tokens = merged_tokens

    return tokens

In [16]:
vocab, merges = train_bpe(corpus, num_merges=50)

test_words = ["abdullah", "wonderland", "rabbit", "curious", "unseenword"]

for w in test_words:
    print(w, "->", tokenize_word_bpe(w, merges))

abdullah -> ['a', 'b', 'd', 'u', 'l', 'l', 'a', 'h', '</w>']
wonderland -> ['w', 'on', 'd', 'er', 'l', 'and</w>']
rabbit -> ['r', 'a', 'b', 'b', 'it</w>']
curious -> ['c', 'u', 'r', 'i', 'ou', 's</w>']
unseenword -> ['u', 'n', 's', 'e', 'en', 'w', 'or', 'd</w>']


In [17]:
def tokenize_text_bpe(text, merges):

    tokens = []

    words = text.split()

    for word in words:
        word_tokens = tokenize_word_bpe(word, merges)
        tokens.extend(word_tokens)

    return tokens

In [18]:
sample = "Deforestation is the global challenge"
tokens = tokenize_text_bpe(sample.lower(), merges)

tokens

['d',
 'e',
 'f',
 'or',
 'e',
 's',
 't',
 'a',
 't',
 'i',
 'on</w>',
 'i',
 's</w>',
 'the</w>',
 'g',
 'l',
 'o',
 'b',
 'al',
 '</w>',
 'ch',
 'al',
 'l',
 'en',
 'g',
 'e</w>']